In [28]:
import pandas as pd 
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [29]:
df = pd.read_csv('final_dataset_2011_2019.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

In [30]:
df['trade_deficit_next'] = df['trade_deficit'].shift(-1)

In [31]:
lags = [1, 2, 3, 6, 12]

for lag in lags:
    df[f'td_lag_{lag}'] = df['trade_deficit'].shift(lag)
    df[f'import_lag_{lag}'] = df['imports_russia'].shift(lag)

In [32]:
df['td_roll_mean_3'] = df['trade_deficit'].shift(1).rolling(3).mean()
df['td_roll_std_3'] = df['trade_deficit'].shift(1).rolling(3).std()

In [33]:
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

In [34]:
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

In [35]:
df['usd_squared'] = df['usd_inr'] ** 2          # nonlinear effect
df['oil_squared'] = df['oil_price'] ** 2

In [36]:
df = df.dropna().reset_index(drop=True)

In [37]:
train = df[df['date'] < '2018-01-01']
test  = df[df['date'] >= '2018-01-01']

In [38]:
features = [col for col in df.columns if col not in ['date', 'trade_deficit_next']]

X_train = train[features]
y_train = train['trade_deficit_next']

X_test = test[features]
y_test = test['trade_deficit_next']

## Baseline Model

In [39]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)

In [40]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("MAE:", mae)
print("RMSE:", rmse)
print("R² Score (Accuracy):", r2)

MAE: 10204731266.028488
RMSE: 10983306543.430365
R² Score (Accuracy): -274.19621601381556


In [41]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

In [42]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8
)
xgb.fit(X_train, y_train)

xgb_preds = xgb.predict(X_test)

In [43]:
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name} -> MAE: {mae:.2f}, RMSE: {rmse:.2f}")

evaluate(y_test, preds, "Linear")
evaluate(y_test, rf_preds, "Random Forest")
evaluate(y_test, xgb_preds, "XGBoost")

Linear -> MAE: 10204731266.03, RMSE: 10983306543.43
Random Forest -> MAE: 507573379.32, RMSE: 692229385.36
XGBoost -> MAE: 548377123.59, RMSE: 701104431.93
